In [ ]:
# ============================================================
# CELL 1 — Import all required libraries
# We import everything we need at the top so it's organized
# ============================================================

import pandas as pd          # for data loading and manipulation
import numpy as np           # for numerical operations
import matplotlib.pyplot as plt  # for plotting charts
import seaborn as sns        # for styled visualizations
import warnings              # to suppress unnecessary warnings

# Hide warnings so output stays clean
warnings.filterwarnings('ignore')

# Set seaborn style globally — all charts will look clean
sns.set_theme(style='whitegrid')

# Make all matplotlib charts display inside the notebook
%matplotlib inline

print("All libraries imported successfully!")

✅ All libraries imported successfully!


In [3]:
# ============================================================
# CELL 2 — Load the raw dataset
# We use pandas read_csv to load our file into a DataFrame
# pd.read_csv reads the CSV and stores it as a table (df)
# ============================================================

# Load the raw CSV file — adjust path if needed
df = pd.read_csv('../data/raw/startup_funding.csv', encoding='latin1')
# Note: encoding='latin1' handles special characters in Indian names

# Display the first 5 rows to get a first look at the data
df.head()

,ï»¿Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJUâS,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [4]:
# ============================================================
# CELL 3 — Dataset overview
# Before cleaning anything, we need to understand:
# - How many rows and columns do we have?
# - What are the data types of each column?
# - How many missing values are there?
# ============================================================

# Shape tells us (rows, columns)
print(f" Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("=" * 55)

# Column names — we'll rename these shortly
print("\nColumn Names:")
for col in df.columns:
    print(f"   → {col}")

print("=" * 55)

# Data types + non-null counts
print("\nData Types & Non-Null Counts:")
print(df.info())

print("=" * 55)

# Missing values per column
print("\nMissing Values per Column:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

 Dataset Shape: 3044 rows × 10 columns

Column Names:
   → ï»¿Sr No
   → Date dd/mm/yyyy
   → Startup Name
   → Industry Vertical
   → SubVertical
   → City  Location
   → Investors Name
   → InvestmentnType
   → Amount in USD
   → Remarks

Data Types & Non-Null Counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   ï»¿Sr No           3044 non-null   int64 
 1   Date dd/mm/yyyy    3044 non-null   object
 2   Startup Name       3044 non-null   object
 3   Industry Vertical  2873 non-null   object
 4   SubVertical        2108 non-null   object
 5   City  Location     2864 non-null   object
 6   Investors Name     3020 non-null   object
 7   InvestmentnType    3040 non-null   object
 8   Amount in USD      2084 non-null   object
 9   Remarks            419 non-null    object
dtypes: int64(1), object(9)
memory usage: 237.9+ KB
None



In [5]:
# ============================================================
# CELL 4 — Rename columns and drop useless ones
# Column names have spaces, typos, and special characters
# Clean names make the rest of the code much easier to write
# ============================================================

# Rename all columns to clean, consistent snake_case names
df.columns = [
    'sr_no',            # serial number
    'date',             # funding date
    'startup_name',     # name of the startup
    'industry',         # industry/sector
    'sub_vertical',     # sub-category within industry
    'city',             # city where startup is based
    'investors',        # investor names
    'investment_type',  # funding round type (Seed, Series A, etc.)
    'amount_usd',       # funding amount in USD
    'remarks'           # 86% missing - useless column
]

# Drop columns we don't need
# 'remarks' is 86% empty - no analytical value
# 'sr_no' is just a row number - pandas index handles this
df.drop(columns=['remarks', 'sr_no'], inplace=True)

# Confirm the result
print("Columns after cleaning:")
print(df.columns.tolist())
print(f"\nShape now: {df.shape[0]} rows x {df.shape[1]} columns")
print("\nFirst 3 rows:")
df.head(3)

Columns after cleaning:
['date', 'startup_name', 'industry', 'sub_vertical', 'city', 'investors', 'investment_type', 'amount_usd']

Shape now: 3044 rows x 8 columns

First 3 rows:


,date,startup_name,industry,sub_vertical,city,investors,investment_type,amount_usd
0,09/01/2020,BYJUâS,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000"
1,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394"
2,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860"


In [6]:
# ============================================================
# CELL 5 — Fix the date column
# Currently 'date' is a plain string like "09/01/2020"
# We need to convert it to a real datetime object so we can
# extract year, month, and do time-based analysis later
# ============================================================

# Convert date column to datetime format
# dayfirst=True because our format is dd/mm/yyyy
# errors='coerce' turns any invalid/unreadable dates into NaT
#   (NaT = Not a Time — pandas version of NaN for dates)
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

# Extract year and month as separate columns
# These will be very useful for trend charts later
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Check how many dates failed to parse (became NaT)
bad_dates = df['date'].isnull().sum()
print(f"Dates that could not be parsed: {bad_dates}")

# Check the year range — should be roughly 2015 to 2020
print(f"\nYear range in dataset:")
print(df['year'].value_counts().sort_index())

Dates that could not be parsed: 8

Year range in dataset:
year
2015.0    929
2016.0    993
2017.0    687
2018.0    309
2019.0    111
2020.0      7
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 6 — Clean the amount_usd column
# Problems we saw:
#   1. Numbers have commas eg. "20,00,00,000" — not a number
#   2. Some values are "undisclosed" or other text strings
#   3. 31% values are missing (NaN)
# Plan:
#   - Remove all commas
#   - Convert to float
#   - Any non-numeric value becomes NaN automatically
# ============================================================

# First let's see what non-numeric values exist in this column
print("Sample of unique non-numeric values in amount_usd:")
non_numeric = df['amount_usd'][
    pd.to_numeric(df['amount_usd'].astype(str).str.replace(',', ''), errors='coerce').isnull()
].unique()
print(non_numeric[:20])  # show first 20 unique bad values

print("=" * 55)

# Step 1 — Remove all commas from the amount column
# "20,00,00,000" becomes "2000000000"
df['amount_usd'] = df['amount_usd'].astype(str).str.replace(',', '', regex=False)

# Step 2 — Convert to numeric float
# errors='coerce' turns "undisclosed" or any text into NaN
df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')

# Step 3 — Check result
print(f"\nMissing values in amount_usd after cleaning: {df['amount_usd'].isnull().sum()}")
print(f"\nBasic stats of amount_usd:")
print(df['amount_usd'].describe())

Sample of unique non-numeric values in amount_usd:
['undisclosed' 'unknown' 'Undisclosed' '14,342,000+' nan
 '\\\\xc2\\\\xa020,000,000' '\\\\xc2\\\\xa016,200,000' '\\\\xc2\\\\xa0N/A'
 '\\\\xc2\\\\xa0600,000' '\\\\xc2\\\\xa0685,000'
 '\\\\xc2\\\\xa019,350,000' '\\\\xc2\\\\xa05,000,000'
 '\\\\xc2\\\\xa010,000,000']

Missing values in amount_usd after cleaning: 979

Basic stats of amount_usd:
count    2.065000e+03
mean     1.842990e+07
std      1.213734e+08
min      1.600000e+04
25%      4.700000e+05
50%      1.700000e+06
75%      8.000000e+06
max      3.900000e+09
Name: amount_usd, dtype: float64


In [8]:
# ============================================================
# CELL 7 — Recover unicode-corrupted values + drop bad rows
# Some amounts have invisible unicode chars (\xc2\xa0) 
# stuck before valid numbers eg. "\xc2\xa020,000,000"
# We strip ALL unicode whitespace, then re-convert to float
# After that we drop rows where amount is still missing
# and rows where date failed to parse
# ============================================================

# Step 1 — Reload amount column from scratch and strip unicode
# We go back to the original string, strip unicode whitespace
# then remove commas, then convert to float
df['amount_usd'] = (
    df['amount_usd']
    .astype(str)                          # convert everything to string
    .str.encode('ascii', errors='ignore') # drop non-ascii unicode chars
    .str.decode('ascii')                  # decode back to clean string
    .str.replace(',', '', regex=False)    # remove commas
    .str.strip()                          # remove any spaces
    .replace('', np.nan)                  # empty strings become NaN
)
df['amount_usd'] = pd.to_numeric(df['amount_usd'], errors='coerce')

# Step 2 — Drop rows where date could not be parsed (8 rows)
df = df.dropna(subset=['date'])
print(f"Rows after dropping bad dates: {len(df)}")

# Step 3 — Drop rows where amount_usd is still missing
# 31% is too many to fill with mean — it would distort analysis
# Better to keep only rows with real funding amounts
df = df.dropna(subset=['amount_usd'])
print(f"Rows after dropping missing amounts: {len(df)}")

# Step 4 — Final check
print(f"\nMissing values remaining:")
print(df.isnull().sum())

Rows after dropping bad dates: 3036
Rows after dropping missing amounts: 2059

Missing values remaining:
date                 0
startup_name         0
industry           129
sub_vertical       642
city               133
investors           20
investment_type      3
amount_usd           0
year                 0
month                0
dtype: int64


In [9]:
# ============================================================
# CELL 8 — Fill remaining missing values
# We don't drop these rows because amount and date are clean
# For text columns, we fill missing with 'Unknown'
# so they don't cause errors during groupby or charts
# ============================================================

# Fill missing text columns with 'Unknown'
# This way they show up clearly in analysis instead of breaking
df['industry']        = df['industry'].fillna('Unknown')
df['sub_vertical']    = df['sub_vertical'].fillna('Unknown')
df['city']            = df['city'].fillna('Unknown')
df['investors']       = df['investors'].fillna('Unknown')
df['investment_type'] = df['investment_type'].fillna('Unknown')

# Confirm zero nulls remain
print("Missing values after filling:")
print(df.isnull().sum())

print(f"\nFinal clean dataset: {df.shape[0]} rows x {df.shape[1]} columns")

Missing values after filling:
date               0
startup_name       0
industry           0
sub_vertical       0
city               0
investors          0
investment_type    0
amount_usd         0
year               0
month              0
dtype: int64

Final clean dataset: 2059 rows x 10 columns


In [10]:
# ============================================================
# CELL 9 — Standardize city names
# Same city is written in multiple ways in the dataset
# eg. "Bangalore", "Bengaluru", "bangalore" are all the same
# We map all variations to one standard name
# This is important so groupby gives correct city counts
# ============================================================

# First let's see how many unique cities we have right now
print(f"Unique cities before cleaning: {df['city'].nunique()}")
print("\nTop 20 city values:")
print(df['city'].value_counts().head(20))

Unique cities before cleaning: 87

Top 20 city values:
city
Bangalore     456
Mumbai        397
New Delhi     241
Gurgaon       198
Unknown       133
Bengaluru     126
Chennai        75
Hyderabad      72
Pune           71
Noida          55
Gurugram       43
Ahmedabad      27
Delhi          25
Jaipur         14
Kolkata        10
Goa             7
Vadodara        6
Chandigarh      6
Singapore       6
Indore          5
Name: count, dtype: int64


In [11]:
# ============================================================
# CELL 10 — Standardize city names
# We create a mapping dictionary of all wrong/alternate names
# and replace them with one standard name
# str.strip() removes accidental spaces before mapping
# ============================================================

# Strip extra spaces from city names first
df['city'] = df['city'].str.strip()

# Mapping of all variations to one standard name
city_mapping = {
    # Bangalore variations
    'Bangalore'         : 'Bengaluru',
    'bangalore'         : 'Bengaluru',
    'Bangaluru'         : 'Bengaluru',

    # Delhi variations
    'New Delhi'         : 'Delhi',
    'new delhi'         : 'Delhi',
    'Delhi'             : 'Delhi',

    # Gurgaon variations
    'Gurgaon'           : 'Gurugram',
    'gurgaon'           : 'Gurugram',

    # Noida — keep as is but clean case
    'noida'             : 'Noida',

    # Mumbai
    'mumbai'            : 'Mumbai',
}

# Apply the mapping
# map() replaces values found in dict, fillna keeps unchanged ones
df['city'] = df['city'].map(city_mapping).fillna(df['city'])

# Confirm results
print(f"Unique cities after cleaning: {df['city'].nunique()}")
print("\nTop 15 cities after standardization:")
print(df['city'].value_counts().head(15))

Unique cities after cleaning: 84

Top 15 cities after standardization:
city
Bengaluru     582
Mumbai        397
Delhi         266
Gurugram      241
Unknown       133
Chennai        75
Hyderabad      72
Pune           71
Noida          55
Ahmedabad      27
Jaipur         14
Kolkata        10
Goa             7
Singapore       6
Chandigarh      6
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 11 — Standardize investment types
# Same funding round is written in many ways
# eg. "Seed Funding", "Seed Round", "seed" are all Seed
# We group them into clean standard categories
# ============================================================

# First see what we are working with
print("Unique investment types before cleaning:")
print(df['investment_type'].value_counts())

Unique investment types before cleaning:
investment_type
Private Equity                 1065
Seed Funding                    711
Seed/ Angel Funding              48
Seed / Angel Funding             38
Debt Funding                     24
Series A                         22
Seed\\nFunding                   22
Series B                         20
Seed/Angel Funding               18
Series C                         14
Series D                         12
Seed Round                        7
Angel / Seed Funding              4
Pre-Series A                      4
Seed                              4
Private Equity Round              4
Unknown                           3
Series F                          2
pre-Series A                      2
Equity                            2
Seed / Angle Funding              2
Venture Round                     2
Series E                          2
Pre-series A                      1
Series G                          1
Series H                          1
pre-ser

In [13]:
# ============================================================
# CELL 12 — Standardize investment types
# We strip whitespace and newline characters first
# Then map all variations to clean standard category names
# ============================================================

# Strip spaces and replace newline characters
df['investment_type'] = df['investment_type'].str.strip()
df['investment_type'] = df['investment_type'].str.replace('\n', ' ', regex=False)
df['investment_type'] = df['investment_type'].str.replace('\\n', ' ', regex=False)

# Mapping all variations to standard names
investment_mapping = {
    # Seed variations
    'Seed Funding'          : 'Seed',
    'Seed Round'            : 'Seed',
    'Seed  Funding'         : 'Seed',
    'Seed/ Angel Funding'   : 'Seed/Angel',
    'Seed / Angel Funding'  : 'Seed/Angel',
    'Seed/Angel Funding'    : 'Seed/Angel',
    'Seed / Angle Funding'  : 'Seed/Angel',
    'Angel / Seed Funding'  : 'Seed/Angel',

    # Private Equity variations
    'Private Equity Round'  : 'Private Equity',
    'Private  Equity'       : 'Private Equity',

    # Pre Series A variations
    'Pre-Series A'          : 'Pre-Series A',
    'pre-Series A'          : 'Pre-Series A',

    # Others
    'Debt Funding'          : 'Debt',
    'Venture Round'         : 'Venture',
    'Crowd Funding'         : 'Crowdfunding',
    'Crowd funding'         : 'Crowdfunding',
    'Equity'                : 'Private Equity',
}

df['investment_type'] = df['investment_type'].map(investment_mapping).fillna(df['investment_type'])

# Confirm
print("Investment types after standardization:")
print(df['investment_type'].value_counts())

Investment types after standardization:
investment_type
Private Equity                 1071
Seed                            722
Seed/Angel                      110
Debt                             24
Series A                         22
Seed\ Funding                    22
Series B                         20
Series C                         14
Series D                         12
Pre-Series A                      6
Unknown                           3
Series F                          2
Venture                           2
Crowdfunding                      2
Series E                          2
Pre-series A                      1
pre-series A                      1
Maiden Round                      1
Single Venture                    1
Seed Funding Round                1
Corporate Round                   1
Funding Round                     1
Series H                          1
Series G                          1
Angel Round                       1
Series J                          1
Angel   

In [14]:
# ============================================================
# CELL 13 — Fix remaining stragglers
# A few values still slipped through with backslashes
# and inconsistent casing — we catch them here
# ============================================================

# Additional mapping for remaining unclean values
stragglers = {
    'Seed\\ Funding'        : 'Seed',
    'Seed Funding Round'    : 'Seed',
    'Pre-series A'          : 'Pre-Series A',
    'pre-series A'          : 'Pre-Series A',
    'PrivateEquity'         : 'Private Equity',
    'Private\\ Equity'      : 'Private Equity',
    'Maiden Round'          : 'Venture',
    'Single Venture'        : 'Venture',
    'Funding Round'         : 'Venture',
    'Corporate Round'       : 'Private Equity',
    'Structured Debt'       : 'Debt',
    'Series H'              : 'Series D+',
    'Series F'              : 'Series D+',
    'Series E'              : 'Series D+',
}

df['investment_type'] = df['investment_type'].map(stragglers).fillna(df['investment_type'])

# Final clean count
print("Final investment types:")
print(df['investment_type'].value_counts())
print(f"\nUnique types now: {df['investment_type'].nunique()}")

Final investment types:
investment_type
Private Equity                 1074
Seed                            745
Seed/Angel                      110
Debt                             25
Series A                         22
Series B                         20
Series C                         14
Series D                         12
Pre-Series A                      8
Venture                           5
Series D+                         5
Unknown                           3
Crowdfunding                      2
Series G                          1
Series J                          1
Angel Round                       1
Venture - Series Unknown          1
Debt and Preference capital       1
Angel                             1
Inhouse Funding                   1
Debt-Funding                      1
Series B (Extension)              1
Mezzanine                         1
Equity Based Funding              1
Private Funding                   1
Private                           1
Term Loan               

In [15]:
# ============================================================
# CELL 14 — Export cleaned dataset to /data/cleaned/
# This is our final clean file that all future notebooks
# and the dashboard will use — we never touch raw data again
# ============================================================

# Define export path
output_path = '../data/cleaned/startup_funding_cleaned.csv'

# Export to CSV — index=False means don't write row numbers
df.to_csv(output_path, index=False)

# Confirm export
print("Cleaned dataset exported successfully.")
print(f"Location: {output_path}")
print(f"Final shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nFinal column list:")
print(df.columns.tolist())
print(f"\nSample of cleaned data:")
df.head(3)

Cleaned dataset exported successfully.
Location: ../data/cleaned/startup_funding_cleaned.csv
Final shape: 2059 rows x 10 columns

Final column list:
['date', 'startup_name', 'industry', 'sub_vertical', 'city', 'investors', 'investment_type', 'amount_usd', 'year', 'month']

Sample of cleaned data:


,date,startup_name,industry,sub_vertical,city,investors,investment_type,amount_usd,year,month
0,2020-01-09,BYJUâS,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity,200000000.0,2020.0,1.0
1,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,8048394.0,2020.0,1.0
2,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,18358860.0,2020.0,1.0
